
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Tracing for Production Agents
#### Duration: 30 minutes
## Overview
This notebook focuses on advanced MLflow Tracing techniques for production agents. You will store traces in Unity Catalog, instrument an agent with manual span decorators and typed spans, attach production-relevant metadata to traces, group multi-turn conversations with session IDs, and search traces with Python and SQL for debugging and analysis.

## Learning Objectives
By the end of this demo, you will be able to:
1. Store an experiment's traces in Unity Catalog tables
2. Create manual spans with `@mlflow.trace` and assign semantic span types
3. Attach custom span attributes for production context (app version, turn number) and trace metadata for session/user grouping
4. Group multi-turn conversations using `mlflow.update_current_trace()` with session-level trace metadata
5. Search traces with `mlflow.search_traces()`, including span attribute filters, and query them with SQL

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 6</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V6</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Install Dependencies and Setup

In [0]:
%run ./Includes/Classroom-Setup-04

### A1. Automatic Tracing and Client Setup

Automatic tracing with `mlflow.openai.autolog()` captures LLM calls with zero additional code. In this demo, we go further: manual spans, custom attributes, session grouping, and programmatic trace search.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Automatic Tracing</strong>
<p style="margin: 8px 0 0 0; color: #333;">
<code>mlflow.openai.autolog()</code> automatically traces OpenAI-compatible LLM calls. Automatic tracing captures the LLM call itself but not custom logic around it. For tool calls, retrieval steps, or business logic, you need <strong>manual tracing</strong>, which is the focus of this demo.
</p>
</div>
</div>
</div>

In [0]:
import mlflow
from databricks_openai import DatabricksOpenAI

# Enable automatic tracing for OpenAI-compatible clients
mlflow.openai.autolog()

# Call the course's shared Unity Gateway model service
client = DatabricksOpenAI(use_ai_gateway=True)
MODEL_SERVICE = "workspace.agent_deployment.agent-model"

### A2. Store Traces in Unity Catalog

Bind the experiment to a Unity Catalog schema so its traces are stored in Delta tables you can govern and query with SQL. The location is set when the experiment is created and can't be changed later. MLflow writes the traces through a SQL warehouse.

In [0]:
import os
from mlflow.entities.trace_location import UnityCatalog

# MLflow writes traces to Unity Catalog through a SQL warehouse.
# Classroom setup found shared_warehouse and set its ID for you.
print(f"MLFLOW_TRACING_SQL_WAREHOUSE_ID: {os.environ['MLFLOW_TRACING_SQL_WAREHOUSE_ID']}")

# Store this experiment's traces in your schema, in tables that start with prod_tracing
experiment = mlflow.set_experiment(
    experiment_name=f"/Workspace/Users/{username}/production-tracing",
    trace_location=UnityCatalog(
        catalog_name=catalog_name,
        schema_name=schema_name,
        table_prefix="prod_tracing",
    ),
)
experiment_id = experiment.experiment_id

print(f"Experiment: {experiment.name}")
print(f"Trace tables: {catalog_name}.{schema_name}.prod_tracing_*")

## B. Manual Tracing with Decorators

### B1. Adding Custom Spans

The `span_type` parameter on `@mlflow.trace` is a semantic label that tells MLflow what kind of GenAI step a function represents. This enables structured trace trees, specialized UI rendering, and downstream evaluation scorers that key off span type.

**NOTE:** `span_type` accepts both string literals (e.g., `"TOOL"`) and the `SpanType` enum (e.g., `SpanType.TOOL`).

#### Common values for <code>span_type</code>:

<br></br>

<div style="width: 100%; font-family: sans-serif;">

<style>

.span-row { padding:4px 6px 8px; box-sizing:border-box; display:grid; grid-template-columns:repeat(auto-fit, minmax(140px, 1fr)); gap:12px; }
.span-box { display:block; background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:14px 16px 16px; cursor:pointer; user-select:none; text-align:left; transition:transform 0.12s, box-shadow 0.15s, border-color 0.15s; }
.span-box::before { content:""; display:block; height:4px; border-radius:2px; background:var(--pc); margin:-2px 0 12px; }
.span-box:hover { transform:translateY(-2px); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.span-box.active { border-color:var(--pc); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.span-title { display:block; font-family:ui-monospace,Menlo,Consolas,monospace; font-size:12.5pt; font-weight:700; color:#0b2026; line-height:1.3; overflow-wrap:anywhere; pointer-events:none; }
.span-detail-wrap { padding:0 6px 8px; box-sizing:border-box; overflow:hidden; max-height:0; opacity:0; margin-top:0; transition:max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; }
.span-detail-wrap.open { max-height:2000px; opacity:1; margin-top:16px; }
.span-detail-card { background:#fff; border:1.5px solid #C6DCEA; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:18px 22px 20px; box-sizing:border-box; }
.span-can-block { font-size:14.5pt; line-height:1.6; color:#0b2026; margin-bottom:14px; }
.span-can-block ul { margin:8px 0 0 0; padding-left:22px; }
.span-can-block li { margin-bottom:8px; }
.span-example-block { background:#F9F7F4; border-left:4px solid; border-radius:0 6px 6px 0; padding:12px 14px; font-size:14pt; line-height:1.5; color:#0b2026; overflow-wrap:anywhere; }

</style>

<div>

  <div class="span-row">
    <span class="span-box active" data-id="0" onclick="selectSpan(0)" style="--pc:#2574B5;">
      <span class="span-title">AGENT</span>
    </span><span class="span-box" data-id="1" onclick="selectSpan(1)" style="--pc:#02A36F;">
      <span class="span-title">TOOL</span>
    </span><span class="span-box" data-id="2" onclick="selectSpan(2)" style="--pc:#FE3722;">
      <span class="span-title">RETRIEVER</span>
    </span><span class="span-box" data-id="3" onclick="selectSpan(3)" style="--pc:#F8A805;">
      <span class="span-title">CHAT_MODEL</span>
    </span><span class="span-box" data-id="4" onclick="selectSpan(4)" style="--pc:#98182A;">
      <span class="span-title">PARSER</span>
    </span><span class="span-box" data-id="5" onclick="selectSpan(5)" style="--pc:#1B5162;">
      <span class="span-title">CHAIN</span>
    </span>
  </div>

  <div class="span-detail-wrap" id="span-detail-wrap">
    <div class="span-detail-card" id="span-detail-card"></div>
  </div>

</div>

</div>

<script>
var SPAN_TYPES = [
  {
    title: "AGENT",
    color: "#2574B5",
    bg: "rgba(37,116,181,0.10)",
    exampleBg: "rgba(37,116,181,0.10)",
    content: `
      <ul>
        <li>The <strong>top-level orchestrator</strong> span that represents the overall agent logic</li>
        <li>Typically the root or near-root span in a trace tree, containing child spans for tools, LLM calls, and other steps</li>
        <li>Use this for the main entry point of your agent pipeline</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="AGENT")<br>def travel_agent(user_query: str) -> str:</code>'
  },
  {
    title: "TOOL",
    color: "#02A36F",
    bg: "rgba(2,163,111,0.10)",
    exampleBg: "rgba(2,163,111,0.10)",
    content: `
      <ul>
        <li>Marks a function as a <strong>tool call</strong>: UC functions, SQL queries, API calls, or any utility the agent invokes</li>
        <li>MLflow renders tool spans with dedicated input/output formatting in the trace UI</li>
        <li>Also available as <code>SpanType.TOOL</code></li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="TOOL")<br>def lookup_weather(city: str) -> dict:</code>'
  },
  {
    title: "RETRIEVER",
    color: "#FE3722",
    bg: "rgba(254,55,34,0.10)",
    exampleBg: "rgba(254,55,34,0.10)",
    content: `
      <ul>
        <li>Represents a <strong>retrieval or AI search</strong> step in a RAG pipeline</li>
        <li>Required by evaluation scorers like <strong>RetrievalGroundedness</strong> and <strong>RetrievalRelevance</strong>. These scorers locate RETRIEVER spans to extract retrieved documents</li>
        <li>Use this for any function that fetches context from an AI search index or document retrieval system</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="RETRIEVER")<br>def search_knowledge_base(query: str) -> list:</code>'
  },
  {
    title: "CHAT_MODEL",
    color: "#F8A805",
    bg: "rgba(248,168,5,0.10)",
    exampleBg: "rgba(248,168,5,0.10)",
    content: `
      <ul>
        <li>Represents the underlying <strong>LLM / chat completion call</strong></li>
        <li>Autologging (e.g., <code>mlflow.openai.autolog()</code>) automatically creates CHAT_MODEL spans for supported providers</li>
        <li>Use this when wrapping a custom or direct LLM call that autologging does not capture</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="CHAT_MODEL")<br>def call_llm(messages: list) -> str:</code>'
  },
  {
    title: "PARSER",
    color: "#98182A",
    bg: "rgba(152,24,42,0.10)",
    exampleBg: "rgba(152,24,42,0.10)",
    content: `
      <ul>
        <li>Marks <strong>parsing, formatting, or pre/post-processing</strong> logic</li>
        <li>Use for functions that transform LLM output into structured data, extract fields from responses, or prepare inputs for downstream steps</li>
        <li>Helps distinguish data transformation from business logic in complex traces</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="PARSER")<br>def extract_json_from_response(raw: str) -> dict:</code>'
  },
  {
    title: "CHAIN / FUNCTION",
    color: "#1B5162",
    bg: "rgba(27,81,98,0.10)",
    exampleBg: "rgba(27,81,98,0.10)",
    content: `
      <ul>
        <li>General-purpose span types for <strong>higher-level chains</strong> or <strong>generic helper functions</strong> in a pipeline</li>
        <li><code>CHAIN</code> is useful for grouping a sequence of steps (e.g., retrieve → process → respond) under a single logical unit</li>
        <li><code>FUNCTION</code> is the most generic type. Use it when none of the other span types apply</li>
      </ul>
    `,
    example: '<code>@mlflow.trace(span_type="CHAIN")<br>def rag_pipeline(question: str) -> str:</code>'
  }
];

var spanCurrent = null;

function selectSpan(id) {
  var wrap = document.getElementById('span-detail-wrap');
  var card = document.getElementById('span-detail-card');
  var c = SPAN_TYPES[id];

  document.querySelectorAll('.span-box').forEach(function(b) {
    var isActive = parseInt(b.dataset.id, 10) === id;
    b.classList.toggle('active', isActive);
  });

  if (spanCurrent === id) {
    wrap.classList.remove('open');
    document.querySelectorAll('.span-box').forEach(function(b) {
      b.classList.remove('active');
    });
    spanCurrent = null;
    return;
  }

  spanCurrent = id;

  card.innerHTML =
    '<div style="height:4px;border-radius:2px;margin:-2px 0 14px;background:' + c.color + ';"></div>'
    + '<div style="font-family:ui-monospace,Menlo,Consolas,monospace;font-size:17pt;font-weight:700;margin-bottom:12px;color:#0b2026;">' + c.title + '</div>'
    + '<div class="span-can-block">'
    + c.content
    + '</div>'
    + '<div class="span-example-block" style="border-color:' + c.color + ';">'
    + '<strong>Usage:</strong> ' + c.example
    + '</div>';

  wrap.classList.add('open');
}

selectSpan(0);
</script>


#### Select each concept to highlight it in the code

<br></br>

<div style="width: 100%; font-family: sans-serif; color: #0b2026;">

<style>

.c1c-cards { padding:4px 6px; box-sizing:border-box; display:grid; grid-template-columns:repeat(auto-fit, minmax(170px, 1fr)); gap:12px; margin-bottom:16px; }
.c1c-card { background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:12px 16px 14px; cursor:pointer; user-select:none; text-align:left; transition:transform 0.12s, box-shadow 0.15s, border-color 0.15s; }
.c1c-card::before { content:""; display:block; height:4px; border-radius:2px; background:var(--dc); margin:-2px 0 10px; }
.c1c-card:hover { transform:translateY(-2px); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.c1c-card.active { border-color:var(--dc); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.c1c-card-label { display:block; font-size:13pt; font-weight:700; color:#0b2026; line-height:1.3; pointer-events:none; overflow-wrap:anywhere; }
.c1c-layout { padding:0 6px 8px; box-sizing:border-box; display:grid; grid-template-columns:minmax(260px, 2fr) 3fr; gap:16px; align-items:start; }
.c1c-explain { order:-1; min-width:0; }
.c1c-code-wrap { min-width:0; }
.c1c-code { margin:0; padding:1rem 0.6rem; overflow-x:hidden !important; font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace; font-size:0.82rem; line-height:1.55; color:#1B3139; background:transparent; }
.c1c-code span { font-family:inherit; }
.c1c-code .tk-kw { color:#a626a4; } .c1c-code .tk-fn { color:#4078f2; } .c1c-code .tk-str { color:#50a14f; } .c1c-code .tk-num { color:#986801; } .c1c-code .tk-cmt { color:#a0a1a7; font-style:italic; } .c1c-code .tk-dim { color:#696c77; } .c1c-code .tk-key { color:#e45649; }
.c1c-code .line { display:block; padding:1px 8px; border-left:3px solid transparent; border-radius:0 3px 3px 0; overflow-wrap:anywhere; transition:background 0.25s, opacity 0.25s; }
.c1c-code.has-highlight .line { opacity:0.32; }
.c1c-code.has-highlight .line.hl { opacity:1; background:rgba(27,81,98,.07); border-left-color:#1B5162; }
.c1c-explain-card { background:#fff; border:1.5px solid #C6DCEA; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:16px 20px 18px; font-size:13pt; line-height:1.6; color:#333; overflow-wrap:anywhere; opacity:0; transition:opacity 0.3s; }
.c1c-explain-card.visible { opacity:1; }
.c1c-explain-card code { overflow-wrap:anywhere; white-space:normal; }
@media (max-width: 820px) { .c1c-layout { grid-template-columns:1fr; } }

</style>

<div class="c1c-cards">
  <div class="c1c-card" data-id="0" onclick="c1cSelect(0)" style="--dc:#02A36F;">
    <span class="c1c-card-label">TOOL Span</span>
  </div>
  <div class="c1c-card" data-id="1" onclick="c1cSelect(1)" style="--dc:#9B59B6;">
    <span class="c1c-card-label">CHAIN Span</span>
  </div>
  <div class="c1c-card" data-id="2" onclick="c1cSelect(2)" style="--dc:#F8A805;">
    <span class="c1c-card-label">Agent Orchestration</span>
  </div>
  <div class="c1c-card" data-id="3" onclick="c1cSelect(3)" style="--dc:#4299E0;">
    <span class="c1c-card-label">Invocation</span>
  </div>
</div>

<div class="c1c-layout">
  <div class="c1c-code-wrap code-card" id="c1c-code-wrap" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
  <div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
    <div style="display:inline-flex;align-items:center;gap:0.45rem;">
      <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
      <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
    </div>
    <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=Array.prototype.map.call(b.closest('.code-card').querySelectorAll('.line'),function(l){return l.textContent.replace(/\u00a0/g,' ').replace(/\s+$/,'')}).join('\n'),t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
      <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
      <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
      <span class="copy-label">Copy</span>
    </button>
  </div>
    <div class="c1c-code" id="c1c-code" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;background:transparent;">
      <span class="line" data-g="0"><span class="tk-fn">@mlflow.trace</span>(<span class="tk-dim">span_type=</span><span class="tk-str">"TOOL"</span>)</span>
      <span class="line" data-g="0"><span class="tk-kw">def</span> <span class="tk-fn">lookup_weather</span>(city: <span class="tk-dim">str</span>) -> <span class="tk-dim">dict</span>:</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt">"""Simulated weather tool for demonstration."""</span></span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;weather_data = {</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"tokyo"</span>: {<span class="tk-str">"temp_f"</span>: <span class="tk-num">72</span>, ...},</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"london"</span>: {<span class="tk-str">"temp_f"</span>: <span class="tk-num">58</span>, ...},</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;}</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">return</span> weather_data.get(city.lower(), ...)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1"><span class="tk-fn">@mlflow.trace</span>(<span class="tk-dim">span_type=</span><span class="tk-str">"CHAIN"</span>)</span>
      <span class="line" data-g="1"><span class="tk-kw">def</span> <span class="tk-fn">travel_agent</span>(user_query: <span class="tk-dim">str</span>) -> <span class="tk-dim">str</span>:</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt">"""A simple agent that looks up weather before responding."""</span></span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt"># Step 1: Determine if weather is needed</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;weather_info = <span class="tk-fn">lookup_weather</span>(<span class="tk-str">"tokyo"</span>)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt"># Step 2: Call the LLM with context</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;response = client.chat.completions.<span class="tk-fn">create</span>(</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;model=MODEL_SERVICE,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;messages=[...],</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;max_tokens=<span class="tk-num">500</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">return</span> response.choices[<span class="tk-num">0</span>].message.content</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">result = <span class="tk-fn">travel_agent</span>(<span class="tk-str">"Should I pack an umbrella for Tokyo?"</span>)</span>
      <span class="line" data-g="3"><span class="tk-fn">print</span>(result)</span>
    </div>
  </div>
  <div class="c1c-explain">
    <div class="c1c-explain-card" id="c1c-explain-card"></div>
  </div>
</div>

</div>

<script>
var C1C_DATA = [
  { color:'#02A36F', title:'@mlflow.trace(span_type="TOOL")',
    text:'Marks <code>lookup_weather</code> as a <strong>TOOL</strong> span. When the agent calls this function, MLflow records it as a distinct tool invocation in the trace tree with its own inputs (city) and outputs (weather dict).' },
  { color:'#9B59B6', title:'@mlflow.trace(span_type="CHAIN")',
    text:'Wraps <code>travel_agent</code> as a <strong>CHAIN</strong> span, grouping a sequence of steps (tool call + LLM call) under a single logical unit. The TOOL span from <code>lookup_weather</code> and the auto-traced CHAT_MODEL span from the LLM call both nest as children.' },
  { color:'#F8A805', title:'Agent Orchestration Logic',
    text:'The body of <code>travel_agent</code> is the orchestration layer: it calls the weather <strong>tool</strong>, injects the result into the prompt, and sends the enriched prompt to the <strong>LLM</strong>. Each step becomes a visible child span.' },
  { color:'#4299E0', title:'Invoking the Agent',
    text:'Calling <code>travel_agent()</code> triggers the full traced execution. The result is a hierarchical trace: <strong>CHAIN</strong> (travel_agent) &rarr; <strong>TOOL</strong> (lookup_weather) + <strong>CHAT_MODEL</strong> (LLM call).' }
];
var c1cCurrent = null, c1cLight = false;
function c1cToggle() {
  c1cLight = !c1cLight;
}
function c1cSelect(id) {
  var code = document.getElementById('c1c-code'), card = document.getElementById('c1c-explain-card');
  document.querySelectorAll('.c1c-card').forEach(function(b){ b.classList.toggle('active', parseInt(b.dataset.id)===id); });
  if (c1cCurrent === id) { code.classList.remove('has-highlight'); card.classList.remove('visible'); document.querySelectorAll('.c1c-card').forEach(function(b){b.classList.remove('active');}); c1cCurrent=null; return; }
  c1cCurrent = id; var c = C1C_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln){ ln.classList.toggle('hl', ln.dataset.g===String(id)); });
  card.innerHTML = '<div style="height:4px;border-radius:2px;margin:-2px 0 12px;background:' + c.color + ';"></div><div style="font-size:14pt;font-weight:700;margin-bottom:10px;color:#0b2026;">'+c.title+'</div><div>'+c.text+'</div>';
  card.classList.add('visible');
}
c1cSelect(0);
</script>

In [0]:
import mlflow
import json

@mlflow.trace(span_type="TOOL")
def lookup_weather(city: str) -> dict:
    """Simulated weather tool for demonstration."""
    weather_data = {
        "tokyo": {"temp_f": 72, "condition": "partly cloudy", "humidity": 65},
        "london": {"temp_f": 58, "condition": "rainy", "humidity": 80},
        "new york": {"temp_f": 68, "condition": "sunny", "humidity": 45},
    }
    return weather_data.get(city.lower(), {"temp_f": 0, "condition": "unknown", "humidity": 0})


@mlflow.trace(span_type="CHAIN")
def travel_agent(user_query: str) -> str:
    """A simple agent that looks up weather before responding."""
    # Step 1: Determine if weather is needed
    weather_info = lookup_weather("tokyo")

    # Step 2: Call the LLM with context
    response = client.chat.completions.create(
        model=MODEL_SERVICE,
        messages=[
            {"role": "system", "content": "You are a travel assistant. Use the provided weather data in your response."},
            {"role": "user", "content": f"{user_query}\n\nCurrent weather: {json.dumps(weather_info)}"}
        ],
        max_tokens=500
    )

    return response.choices[0].message.content


result = travel_agent("Should I pack an umbrella for Tokyo this weekend?")
print(result)

### B2. Reading the Span Tree

The trace now shows a hierarchy. Each span records its start time, end time, inputs, outputs, and status.

<div style="max-width:680px;margin:16px 0;font-family:'Menlo','Consolas',monospace;font-size:13pt;line-height:2;background:#f7f8fc;border:1px solid #dde0e8;border-radius:10px;padding:18px 22px;">
<div style="color:#3b82f6;font-weight:700;">CHAIN <span style="color:#999;font-weight:400;">travel_agent</span></div>
<div style="margin-left:24px;border-left:2px solid #dde0e8;padding-left:14px;">
<div style="color:#16a34a;font-weight:700;">TOOL <span style="color:#999;font-weight:400;">lookup_weather</span> <span style="color:#bbb;font-size:11pt;">inputs: "tokyo" &rarr; outputs: {temp_f: 72, ...}</span></div>
<div style="color:#a855f7;font-weight:700;">CHAT_MODEL <span style="color:#999;font-weight:400;">auto-captured</span> <span style="color:#bbb;font-size:11pt;">prompt + response</span></div>
</div>
</div>

Use this execution graph to debug how the agent handled each request.

## C. Production Context on Traces

### C1. Adding Metadata to Spans

In production, you need to correlate traces with business context: which user made the request, what session it belongs to, what application version is running. MLflow provides two mechanisms: **trace metadata** for session and user grouping (via `mlflow.update_current_trace()`), and **span attributes** for per-request details (via `span.set_attributes()`).

For example, after running the cell below, check the trace metadata for the session and user fields, and click on **Attributes** to view the span attributes logged with `span.set_attributes()`.


#### Select each concept to highlight it in the code

<br></br>

<div style="width: 100%; font-family: sans-serif; color: #0b2026;">

<style>

.d1-cards { padding:4px 6px; box-sizing:border-box; display:grid; grid-template-columns:repeat(auto-fit, minmax(170px, 1fr)); gap:12px; margin-bottom:16px; }
.d1-card { background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:12px 16px 14px; cursor:pointer; user-select:none; text-align:left; transition:transform 0.12s, box-shadow 0.15s, border-color 0.15s; }
.d1-card::before { content:""; display:block; height:4px; border-radius:2px; background:var(--dc); margin:-2px 0 10px; }
.d1-card:hover { transform:translateY(-2px); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.d1-card.active { border-color:var(--dc); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.d1-card-label { display:block; font-size:13pt; font-weight:700; color:#0b2026; line-height:1.3; pointer-events:none; overflow-wrap:anywhere; }
.d1-layout { padding:0 6px 8px; box-sizing:border-box; display:grid; grid-template-columns:minmax(260px, 2fr) 3fr; gap:16px; align-items:start; }
.d1-explain { order:-1; min-width:0; }
.d1-code-wrap { min-width:0; }
.d1-code { margin:0; padding:1rem 0.6rem; overflow-x:hidden !important; font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace; font-size:0.82rem; line-height:1.55; color:#1B3139; background:transparent; }
.d1-code span { font-family:inherit; }
.d1-code .tk-kw { color:#a626a4; } .d1-code .tk-fn { color:#4078f2; } .d1-code .tk-str { color:#50a14f; } .d1-code .tk-num { color:#986801; } .d1-code .tk-cmt { color:#a0a1a7; font-style:italic; } .d1-code .tk-dim { color:#696c77; } .d1-code .tk-key { color:#e45649; }
.d1-code .line { display:block; padding:1px 8px; border-left:3px solid transparent; border-radius:0 3px 3px 0; overflow-wrap:anywhere; transition:background 0.25s, opacity 0.25s; }
.d1-code.has-highlight .line { opacity:0.32; }
.d1-code.has-highlight .line.hl { opacity:1; background:rgba(27,81,98,.07); border-left-color:#1B5162; }
.d1-explain-card { background:#fff; border:1.5px solid #C6DCEA; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:16px 20px 18px; font-size:13pt; line-height:1.6; color:#333; overflow-wrap:anywhere; opacity:0; transition:opacity 0.3s; }
.d1-explain-card.visible { opacity:1; }
.d1-explain-card code { overflow-wrap:anywhere; white-space:normal; }
@media (max-width: 820px) { .d1-layout { grid-template-columns:1fr; } }

</style>

<!-- Card selectors -->
<div class="d1-cards">
  <div class="d1-card" data-id="0" onclick="d1Select(0)" style="--dc:#9B59B6;">
    <span class="d1-card-label">@mlflow.trace</span>
  </div>
  <div class="d1-card" data-id="1" onclick="d1Select(1)" style="--dc:#4299E0;">
    <span class="d1-card-label">Trace Metadata</span>
  </div>
  <div class="d1-card" data-id="2" onclick="d1Select(2)" style="--dc:#02A36F;">
    <span class="d1-card-label">Span Attributes</span>
  </div>
  <div class="d1-card" data-id="3" onclick="d1Select(3)" style="--dc:#F8A805;">
    <span class="d1-card-label">Tool + LLM</span>
  </div>
</div>

<!-- Code + explanation side by side -->
<div class="d1-layout">

  <div class="d1-code-wrap code-card" id="d1-code-wrap" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
  <div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
    <div style="display:inline-flex;align-items:center;gap:0.45rem;">
      <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
      <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
    </div>
    <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=Array.prototype.map.call(b.closest('.code-card').querySelectorAll('.line'),function(l){return l.textContent.replace(/\u00a0/g,' ').replace(/\s+$/,'')}).join('\n'),t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
      <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
      <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
      <span class="copy-label">Copy</span>
    </button>
  </div>
    <div class="d1-code" id="d1-code" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;background:transparent;">
      <span class="line" data-g="0"><span class="tk-fn">@mlflow.trace</span>(<span class="tk-dim">span_type=</span><span class="tk-str">"CHAIN"</span>)</span>
      <span class="line" data-g="0"><span class="tk-kw">def</span> <span class="tk-fn">travel_agent_with_context</span>(user_query, user_id, session_id):</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt">"""Agent with production context attached to the trace."""</span></span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt"># Session/user go on the trace for grouping</span></span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;mlflow.<span class="tk-fn">update_current_trace</span>(<span class="tk-dim">session_id=</span>session_id, <span class="tk-dim">user=</span>user_id)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt"># Per-request details stay as span attributes</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;span = mlflow.<span class="tk-fn">get_current_active_span</span>()</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;span.<span class="tk-fn">set_attributes</span>({</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"app_version"</span>: <span class="tk-str">"1.2.0"</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"request_source"</span>: <span class="tk-str">"web_ui"</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;})</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;weather_info = <span class="tk-fn">lookup_weather</span>(<span class="tk-str">"london"</span>)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;response = client.chat.completions.<span class="tk-fn">create</span>(</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;model=MODEL_SERVICE,</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;messages=[...],</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;max_tokens=<span class="tk-num">500</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;)</span>
      <span class="line">&nbsp;</span>
      <span class="line">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">return</span> response.choices[<span class="tk-num">0</span>].message.content</span>
    </div>
  </div>

  <div class="d1-explain">
    <div class="d1-explain-card" id="d1-explain-card"></div>
  </div>

</div>

</div>

<script>
var D1_DATA = [
  {
    color: '#9B59B6',
    title: '@mlflow.trace(span_type="CHAIN")',
    text: 'The decorator wraps the entire function as a <strong>CHAIN</strong> span. Everything that executes inside this function becomes a child in the trace tree. The <code>span_type</code> label tells MLflow this is a higher-level orchestration step, not a tool or LLM call.'
  },
  {
    color: '#4299E0',
    title: 'update_current_trace()',
    text: '<code>mlflow.update_current_trace()</code> sets <code>session_id</code> and <code>user</code> as <strong>trace metadata</strong>. MLflow uses these to group traces in the Sessions view and to filter by user — they must be trace metadata, not span attributes.'
  },
  {
    color: '#02A36F',
    title: 'Span Attributes',
    text: '<code>set_attributes()</code> attaches key-value pairs to the span. Use span attributes for per-request details like <code>app_version</code> and <code>request_source</code> that are <strong>searchable and filterable</strong> in the MLflow UI and via <code>mlflow.search_traces()</code>.'
  },
  {
    color: '#F8A805',
    title: 'Tool Call + LLM Request',
    text: 'The <code>lookup_weather</code> tool call and <code>chat.completions.create</code> LLM call execute as normal Python. Because autologging is enabled, MLflow automatically creates a <strong>TOOL</strong> child span and a <strong>CHAT_MODEL</strong> child span nested under this CHAIN.'
  }
];

var d1Current = null;

function d1Select(id) {
  var codeEl = document.getElementById('d1-code');
  var explainCard = document.getElementById('d1-explain-card');

  // Toggle cards
  document.querySelectorAll('.d1-card').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id);
  });

  // Deselect if same card clicked
  if (d1Current === id) {
    codeEl.classList.remove('has-highlight');
    explainCard.classList.remove('visible');
    document.querySelectorAll('.d1-card').forEach(function(b) { b.classList.remove('active'); });
    d1Current = null;
    return;
  }

  d1Current = id;
  var c = D1_DATA[id];

  // Highlight matching code lines
  codeEl.classList.add('has-highlight');
  codeEl.querySelectorAll('.line').forEach(function(ln) {
    ln.classList.toggle('hl', ln.dataset.g === String(id));
  });

  // Show explanation
  explainCard.innerHTML =
    '<div style="height:4px;border-radius:2px;margin:-2px 0 12px;background:' + c.color + ';"></div><div style="font-size:14pt;font-weight:700;margin-bottom:10px;color:#0b2026;">' + c.title + '</div>'
    + '<div>' + c.text + '</div>';
  explainCard.classList.add('visible');
}
d1Select(0);
</script>

In [0]:
@mlflow.trace(span_type="CHAIN")
def travel_agent_with_context(user_query: str, user_id: str, session_id: str) -> str:
    """Agent with production context attached to the trace."""
    # Session/user go on the trace for grouping in the Sessions view
    mlflow.update_current_trace(session_id=session_id, user=user_id)

    # Per-request details stay as span attributes
    span = mlflow.get_current_active_span()
    span.set_attributes({
        "app_version": "1.2.0",
        "request_source": "web_ui"
    })

    weather_info = lookup_weather("london")

    response = client.chat.completions.create(
        model=MODEL_SERVICE,
        messages=[
            {"role": "system", "content": "You are a travel assistant."},
            {"role": "user", "content": f"{user_query}\n\nWeather: {json.dumps(weather_info)}"}
        ],
        max_tokens=500
    )

    return response.choices[0].message.content


result = travel_agent_with_context(
    user_query="What should I wear in London tomorrow?",
    user_id="user_42",
    session_id="sess_abc123"
)
print(result)

### C2. Why Attributes Matter

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Because this experiment stores its traces in Unity Catalog, you can search span attributes with <code>span.attributes.&amp;lt;key&amp;gt;</code> filters. Use them for per-request details like app version and request source. For session and user grouping, use <code>mlflow.update_current_trace()</code> to set trace metadata, which the Sessions view uses. Attach both while the request runs; you can't add them afterward.</p>
</div>
</div>
</div>

## D. Multi-Turn Conversation Tracing

### D1. Session-Level Grouping

Production agents often handle multi-turn conversations. Each turn produces its own trace, but you need to group them into sessions for analysis. Use `mlflow.update_current_trace()` to set `session_id` and `user` as **trace metadata** on each turn — this is how MLflow groups turns in the Sessions view.


#### Select each concept to highlight it in the code

<br></br>

<div style="width: 100%; font-family: sans-serif; color: #0b2026;">

<style>

.e1-cards { padding:4px 6px; box-sizing:border-box; display:grid; grid-template-columns:repeat(auto-fit, minmax(170px, 1fr)); gap:12px; margin-bottom:16px; }
.e1-card { background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:12px 16px 14px; cursor:pointer; user-select:none; text-align:left; transition:transform 0.12s, box-shadow 0.15s, border-color 0.15s; }
.e1-card::before { content:""; display:block; height:4px; border-radius:2px; background:var(--dc); margin:-2px 0 10px; }
.e1-card:hover { transform:translateY(-2px); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.e1-card.active { border-color:var(--dc); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.e1-card-label { display:block; font-size:13pt; font-weight:700; color:#0b2026; line-height:1.3; pointer-events:none; overflow-wrap:anywhere; }
.e1-layout { padding:0 6px 8px; box-sizing:border-box; display:grid; grid-template-columns:minmax(260px, 2fr) 3fr; gap:16px; align-items:start; }
.e1-explain { order:-1; min-width:0; }
.e1-code-wrap { min-width:0; }
.e1-code { margin:0; padding:1rem 0.6rem; overflow-x:hidden !important; font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace; font-size:0.82rem; line-height:1.55; color:#1B3139; background:transparent; }
.e1-code span { font-family:inherit; }
.e1-code .tk-kw { color:#a626a4; } .e1-code .tk-fn { color:#4078f2; } .e1-code .tk-str { color:#50a14f; } .e1-code .tk-num { color:#986801; } .e1-code .tk-cmt { color:#a0a1a7; font-style:italic; } .e1-code .tk-dim { color:#696c77; } .e1-code .tk-key { color:#e45649; }
.e1-code .line { display:block; padding:1px 8px; border-left:3px solid transparent; border-radius:0 3px 3px 0; overflow-wrap:anywhere; transition:background 0.25s, opacity 0.25s; }
.e1-code.has-highlight .line { opacity:0.32; }
.e1-code.has-highlight .line.hl { opacity:1; background:rgba(27,81,98,.07); border-left-color:#1B5162; }
.e1-explain-card { background:#fff; border:1.5px solid #C6DCEA; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:16px 20px 18px; font-size:13pt; line-height:1.6; color:#333; overflow-wrap:anywhere; opacity:0; transition:opacity 0.3s; }
.e1-explain-card.visible { opacity:1; }
.e1-explain-card code { overflow-wrap:anywhere; white-space:normal; }
@media (max-width: 820px) { .e1-layout { grid-template-columns:1fr; } }

</style>

<div class="e1-cards">
  <div class="e1-card" data-id="0" onclick="e1Select(0)" style="--dc:#9B59B6;">
    <span class="e1-card-label">Session Setup</span>
  </div>
  <div class="e1-card" data-id="1" onclick="e1Select(1)" style="--dc:#4299E0;">
    <span class="e1-card-label">Turn Decorator</span>
  </div>
  <div class="e1-card" data-id="2" onclick="e1Select(2)" style="--dc:#02A36F;">
    <span class="e1-card-label">Trace Metadata</span>
  </div>
  <div class="e1-card" data-id="3" onclick="e1Select(3)" style="--dc:#F8A805;">
    <span class="e1-card-label">Conversation Memory</span>
  </div>
</div>

<div class="e1-layout">
  <div class="e1-code-wrap code-card" id="e1-code-wrap" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
  <div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
    <div style="display:inline-flex;align-items:center;gap:0.45rem;">
      <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
      <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
    </div>
    <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=Array.prototype.map.call(b.closest('.code-card').querySelectorAll('.line'),function(l){return l.textContent.replace(/\u00a0/g,' ').replace(/\s+$/,'')}).join('\n'),t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
      <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
      <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
      <span class="copy-label">Copy</span>
    </button>
  </div>
    <div class="e1-code" id="e1-code" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;background:transparent;">
      <span class="line" data-g="0"><span class="tk-kw">import</span> uuid</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="0">session_id = <span class="tk-fn">str</span>(<span class="tk-fn">uuid.uuid4</span>())</span>
      <span class="line" data-g="0">conversation_history = [</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;{<span class="tk-str">"role"</span>: <span class="tk-str">"system"</span>, <span class="tk-str">"content"</span>: <span class="tk-str">"You are a travel assistant. Be concise."</span>}</span>
      <span class="line" data-g="0">]</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1"><span class="tk-fn">@mlflow.trace</span>(<span class="tk-dim">span_type=</span><span class="tk-str">"CHAIN"</span>, <span class="tk-dim">name=</span><span class="tk-str">"conversation_turn"</span>)</span>
      <span class="line" data-g="1"><span class="tk-kw">def</span> <span class="tk-fn">chat_turn</span>(user_message: <span class="tk-dim">str</span>, turn_number: <span class="tk-dim">int</span>, user_id: <span class="tk-dim">str</span> = <span class="tk-str">"demo_user"</span>) -> <span class="tk-dim">str</span>:</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt"># Session/user go on the trace for session grouping</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;mlflow.<span class="tk-fn">update_current_trace</span>(<span class="tk-dim">session_id=</span>session_id, <span class="tk-dim">user=</span>user_id)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-cmt"># Per-turn details stay as span attributes</span></span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;mlflow.<span class="tk-fn">get_current_active_span</span>().<span class="tk-fn">set_attributes</span>({<span class="tk-str">"turn_number"</span>: turn_number})</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;conversation_history.<span class="tk-fn">append</span>({<span class="tk-str">"role"</span>: <span class="tk-str">"user"</span>, <span class="tk-str">"content"</span>: user_message})</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;response = client.chat.completions.<span class="tk-fn">create</span>(</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;model=MODEL_SERVICE,</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;messages=conversation_history,</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;max_tokens=<span class="tk-num">300</span></span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;assistant_msg = response.choices[<span class="tk-num">0</span>].message.content</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;conversation_history.<span class="tk-fn">append</span>({<span class="tk-str">"role"</span>: <span class="tk-str">"assistant"</span>, <span class="tk-str">"content"</span>: assistant_msg})</span>
      <span class="line" data-g="3">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-kw">return</span> assistant_msg</span>
    </div>
  </div>
  <div class="e1-explain">
    <div class="e1-explain-card" id="e1-explain-card"></div>
  </div>
</div>

</div>

<script>
var E1_DATA = [
  { color:'#9B59B6', title:'Session Setup',
    text:'A single <code>session_id</code> (UUID) is generated once and shared across all turns. The <code>conversation_history</code> list accumulates messages, giving the LLM context from prior turns.' },
  { color:'#4299E0', title:'Decorator with name= Parameter',
    text:'<code>@mlflow.trace(name="conversation_turn")</code> gives the span a <strong>custom name</strong> instead of defaulting to the function name. This makes traces easier to identify when browsing the UI.' },
  { color:'#02A36F', title:'Trace Metadata vs Span Attributes',
    text:'<code>mlflow.update_current_trace()</code> sets <code>session_id</code> and <code>user</code> as <strong>trace metadata</strong> — this is how MLflow groups turns into sessions in the Sessions view. Per-turn details like <code>turn_number</code> stay as span attributes.' },
  { color:'#F8A805', title:'Conversation Memory & LLM Call',
    text:'The user message is appended to history, the full history is sent to the LLM, and the assistant response is appended back. This accumulating pattern is how multi-turn context is maintained. The LLM call is auto-traced as a <strong>CHAT_MODEL</strong> child span.' }
];
var e1Current = null, e1Light = false;
function e1Toggle() {
  e1Light = !e1Light;
}
function e1Select(id) {
  var code = document.getElementById('e1-code'), card = document.getElementById('e1-explain-card');
  document.querySelectorAll('.e1-card').forEach(function(b){ b.classList.toggle('active', parseInt(b.dataset.id)===id); });
  if (e1Current === id) { code.classList.remove('has-highlight'); card.classList.remove('visible'); document.querySelectorAll('.e1-card').forEach(function(b){b.classList.remove('active');}); e1Current=null; return; }
  e1Current = id; var c = E1_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln){ ln.classList.toggle('hl', ln.dataset.g===String(id)); });
  card.innerHTML = '<div style="height:4px;border-radius:2px;margin:-2px 0 12px;background:' + c.color + ';"></div><div style="font-size:14pt;font-weight:700;margin-bottom:10px;color:#0b2026;">'+c.title+'</div><div>'+c.text+'</div>';
  card.classList.add('visible');
}
e1Select(0);
</script>

In [0]:
import uuid

session_id = str(uuid.uuid4())
conversation_history = [
    {"role": "system", "content": "You are a travel assistant. Be concise."}
]


@mlflow.trace(span_type="CHAIN", name="conversation_turn")
def chat_turn(user_message: str, turn_number: int, user_id: str = "demo_user") -> str:
    """Process a single turn in a multi-turn conversation."""
    # Session/user go on the trace, so MLflow can group the turns together
    mlflow.update_current_trace(session_id=session_id, user=user_id)

    # Per-turn details are fine as span attributes
    mlflow.get_current_active_span().set_attributes({"turn_number": turn_number})

    conversation_history.append({"role": "user", "content": user_message})

    response = client.chat.completions.create(
        model=MODEL_SERVICE,
        messages=conversation_history,
        max_tokens=300
    )

    assistant_msg = response.choices[0].message.content
    conversation_history.append({"role": "assistant", "content": assistant_msg})
    return assistant_msg


# Simulate a multi-turn conversation
print("Turn 1:", chat_turn("I'm planning a trip to Japan in April.", turn_number=1))

In [0]:
print("Turn 2:", chat_turn("What about the cherry blossoms, will I catch them?", turn_number=2))

In [0]:
print("Turn 3:", chat_turn("Do you recommend anything else to do while in Kyoto?", turn_number=3))

### D2. Viewing Session Traces

Each turn above produced a separate trace, but they all share the same `session_id` in their trace metadata. MLflow uses this to group the turns in the Sessions view, letting you reconstruct the full conversation during debugging or analysis.

## E. Searching and Filtering Traces

### E1. Programmatic Trace Search

In production, you will not browse traces one at a time in the UI. `mlflow.search_traces()` lets you query traces programmatically. Filter by time range, status, attributes, or any combination. For traces in Unity Catalog, add a time range so each search scans less data. New traces can take a few seconds to appear.


#### Select each concept to highlight it in the code

<br></br>

<div style="width: 100%; font-family: sans-serif; color: #0b2026;">

<style>

.f1-cards { padding:4px 6px; box-sizing:border-box; display:grid; grid-template-columns:repeat(auto-fit, minmax(170px, 1fr)); gap:12px; margin-bottom:16px; }
.f1-card { background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:12px 16px 14px; cursor:pointer; user-select:none; text-align:left; transition:transform 0.12s, box-shadow 0.15s, border-color 0.15s; }
.f1-card::before { content:""; display:block; height:4px; border-radius:2px; background:var(--dc); margin:-2px 0 10px; }
.f1-card:hover { transform:translateY(-2px); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.f1-card.active { border-color:var(--dc); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.f1-card-label { display:block; font-size:13pt; font-weight:700; color:#0b2026; line-height:1.3; pointer-events:none; overflow-wrap:anywhere; }
.f1-layout { padding:0 6px 8px; box-sizing:border-box; display:grid; grid-template-columns:minmax(260px, 2fr) 3fr; gap:16px; align-items:start; }
.f1-explain { order:-1; min-width:0; }
.f1-code-wrap { min-width:0; }
.f1-code { margin:0; padding:1rem 0.6rem; overflow-x:hidden !important; font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace; font-size:0.82rem; line-height:1.55; color:#1B3139; background:transparent; }
.f1-code span { font-family:inherit; }
.f1-code .tk-kw { color:#a626a4; } .f1-code .tk-fn { color:#4078f2; } .f1-code .tk-str { color:#50a14f; } .f1-code .tk-num { color:#986801; } .f1-code .tk-cmt { color:#a0a1a7; font-style:italic; } .f1-code .tk-dim { color:#696c77; } .f1-code .tk-key { color:#e45649; }
.f1-code .line { display:block; padding:1px 8px; border-left:3px solid transparent; border-radius:0 3px 3px 0; overflow-wrap:anywhere; transition:background 0.25s, opacity 0.25s; }
.f1-code.has-highlight .line { opacity:0.32; }
.f1-code.has-highlight .line.hl { opacity:1; background:rgba(27,81,98,.07); border-left-color:#1B5162; }
.f1-explain-card { background:#fff; border:1.5px solid #C6DCEA; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:16px 20px 18px; font-size:13pt; line-height:1.6; color:#333; overflow-wrap:anywhere; opacity:0; transition:opacity 0.3s; }
.f1-explain-card.visible { opacity:1; }
.f1-explain-card code { overflow-wrap:anywhere; white-space:normal; }
@media (max-width: 820px) { .f1-layout { grid-template-columns:1fr; } }

</style>

<div class="f1-cards">
  <div class="f1-card" data-id="0" onclick="f1Select(0)" style="--dc:#9B59B6;">
    <span class="f1-card-label">Time Range</span>
  </div>
  <div class="f1-card" data-id="1" onclick="f1Select(1)" style="--dc:#02A36F;">
    <span class="f1-card-label">Search Traces</span>
  </div>
  <div class="f1-card" data-id="2" onclick="f1Select(2)" style="--dc:#4299E0;">
    <span class="f1-card-label">Display Results</span>
  </div>
</div>

<div class="f1-layout">
  <div class="f1-code-wrap code-card" id="f1-code-wrap" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
  <div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
    <div style="display:inline-flex;align-items:center;gap:0.45rem;">
      <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
      <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
    </div>
    <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=Array.prototype.map.call(b.closest('.code-card').querySelectorAll('.line'),function(l){return l.textContent.replace(/\u00a0/g,' ').replace(/\s+$/,'')}).join('\n'),t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
      <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
      <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
      <span class="copy-label">Copy</span>
    </button>
  </div>
    <div class="f1-code" id="f1-code" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;background:transparent;">
      <span class="line" data-g="0"><span class="tk-kw">import</span> time</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="0"><span class="tk-cmt"># Search only the last hour of traces, so the query scans less data</span></span>
      <span class="line" data-g="0">one_hour_ago_ms = <span class="tk-fn">int</span>(time.<span class="tk-fn">time</span>() * <span class="tk-num">1000</span>) - <span class="tk-num">60</span> * <span class="tk-num">60</span> * <span class="tk-num">1000</span></span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1">traces_df = mlflow.<span class="tk-fn">search_traces</span>(</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;locations=[experiment_id],</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;filter_string=<span class="tk-str">f"trace.timestamp_ms > {one_hour_ago_ms}"</span>,</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;max_results=<span class="tk-num">10</span></span>
      <span class="line" data-g="1">)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2"><span class="tk-fn">display</span>(traces_df[[</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"client_request_id"</span>, <span class="tk-str">"trace_id"</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"request_time"</span>, <span class="tk-str">"state"</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;<span class="tk-str">"execution_duration"</span></span>
      <span class="line" data-g="2">]])</span>
    </div>
  </div>
  <div class="f1-explain">
    <div class="f1-explain-card" id="f1-explain-card"></div>
  </div>
</div>

</div>

<script>
var F1_DATA = [
  { color:'#9B59B6', title:'Limit the Time Range',
    text:'Computes a timestamp one hour ago, in milliseconds. A <code>trace.timestamp_ms</code> filter keeps each search of the Unity Catalog trace tables to recent data, so it runs faster and costs less.' },
  { color:'#02A36F', title:'mlflow.search_traces()',
    text:'The core query API. Takes a <code>locations</code> list (experiment IDs) and optional <code>filter_string</code>, <code>max_results</code>, and <code>order_by</code> parameters. Filter on <code>trace.*</code> (status, timing), <code>metadata.*</code>, <code>tag.*</code>, or <code>span.attributes.*</code>. Returns a <strong>Pandas DataFrame</strong> with one row per trace.' },
  { color:'#4299E0', title:'Display Selected Columns',
    text:'Selects the most useful columns for a quick overview: <code>client_request_id</code>, <code>trace_id</code>, <code>request_time</code>, <code>state</code> (OK or ERROR), and <code>execution_duration</code>. The Databricks <code>display()</code> renders them as an interactive table.' }
];
var f1Current = null, f1Light = false;
function f1Toggle() {
  f1Light = !f1Light;
}
function f1Select(id) {
  var code = document.getElementById('f1-code'), card = document.getElementById('f1-explain-card');
  document.querySelectorAll('.f1-card').forEach(function(b){ b.classList.toggle('active', parseInt(b.dataset.id)===id); });
  if (f1Current === id) { code.classList.remove('has-highlight'); card.classList.remove('visible'); document.querySelectorAll('.f1-card').forEach(function(b){b.classList.remove('active');}); f1Current=null; return; }
  f1Current = id; var c = F1_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln){ ln.classList.toggle('hl', ln.dataset.g===String(id)); });
  card.innerHTML = '<div style="height:4px;border-radius:2px;margin:-2px 0 12px;background:' + c.color + ';"></div><div style="font-size:14pt;font-weight:700;margin-bottom:10px;color:#0b2026;">'+c.title+'</div><div>'+c.text+'</div>';
  card.classList.add('visible');
}
f1Select(0);
</script>

In [0]:
import time

# Search only the last hour of traces, so the query scans less data
one_hour_ago_ms = int(time.time() * 1000) - 60 * 60 * 1000

traces_df = mlflow.search_traces(
    locations=[experiment_id],
    filter_string=f"trace.timestamp_ms > {one_hour_ago_ms}",
    max_results=10
)

display(traces_df[["client_request_id", "trace_id", "request_time", "state", "execution_duration"]])

### E2. Filtering by Status, Timing, and Attributes

Common production queries include finding error traces, slow traces, and traces from a specific app version.


#### Select each concept to highlight it in the code

<br></br>

<div style="width: 100%; font-family: sans-serif; color: #0b2026;">

<style>

.f2-cards { padding:4px 6px; box-sizing:border-box; display:grid; grid-template-columns:repeat(auto-fit, minmax(170px, 1fr)); gap:12px; margin-bottom:16px; }
.f2-card { background:#fff; border:2px solid #C6DCEA; box-sizing:border-box; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:12px 16px 14px; cursor:pointer; user-select:none; text-align:left; transition:transform 0.12s, box-shadow 0.15s, border-color 0.15s; }
.f2-card::before { content:""; display:block; height:4px; border-radius:2px; background:var(--dc); margin:-2px 0 10px; }
.f2-card:hover { transform:translateY(-2px); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.f2-card.active { border-color:var(--dc); box-shadow:0 6px 16px rgba(27,49,57,.10); }
.f2-card-label { display:block; font-size:13pt; font-weight:700; color:#0b2026; line-height:1.3; pointer-events:none; overflow-wrap:anywhere; }
.f2-layout { padding:0 6px 8px; box-sizing:border-box; display:grid; grid-template-columns:minmax(260px, 2fr) 3fr; gap:16px; align-items:start; }
.f2-explain { order:-1; min-width:0; }
.f2-code-wrap { min-width:0; }
.f2-code { margin:0; padding:1rem 0.6rem; overflow-x:hidden !important; font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace; font-size:0.82rem; line-height:1.55; color:#1B3139; background:transparent; }
.f2-code span { font-family:inherit; }
.f2-code .tk-kw { color:#a626a4; } .f2-code .tk-fn { color:#4078f2; } .f2-code .tk-str { color:#50a14f; } .f2-code .tk-num { color:#986801; } .f2-code .tk-cmt { color:#a0a1a7; font-style:italic; } .f2-code .tk-dim { color:#696c77; } .f2-code .tk-key { color:#e45649; }
.f2-code .line { display:block; padding:1px 8px; border-left:3px solid transparent; border-radius:0 3px 3px 0; overflow-wrap:anywhere; transition:background 0.25s, opacity 0.25s; }
.f2-code.has-highlight .line { opacity:0.32; }
.f2-code.has-highlight .line.hl { opacity:1; background:rgba(27,81,98,.07); border-left-color:#1B5162; }
.f2-explain-card { background:#fff; border:1.5px solid #C6DCEA; border-radius:12px; box-shadow:0 2px 8px rgba(27,49,57,.07); padding:16px 20px 18px; font-size:13pt; line-height:1.6; color:#333; overflow-wrap:anywhere; opacity:0; transition:opacity 0.3s; }
.f2-explain-card.visible { opacity:1; }
.f2-explain-card code { overflow-wrap:anywhere; white-space:normal; }
@media (max-width: 820px) { .f2-layout { grid-template-columns:1fr; } }

</style>

<div class="f2-cards">
  <div class="f2-card" data-id="0" onclick="f2Select(0)" style="--dc:#FE3722;">
    <span class="f2-card-label">Error Traces</span>
  </div>
  <div class="f2-card" data-id="1" onclick="f2Select(1)" style="--dc:#F8A805;">
    <span class="f2-card-label">Slow Traces</span>
  </div>
  <div class="f2-card" data-id="2" onclick="f2Select(2)" style="--dc:#2272B4;">
    <span class="f2-card-label">Span Attribute</span>
  </div>
</div>

<div class="f2-layout">
  <div class="f2-code-wrap code-card" id="f2-code-wrap" style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;border:1px solid #DCE0E2;border-radius:8px;overflow:hidden;background:#F9F7F4;max-width:100%;margin:12px 0;">
  <div style="display:flex;align-items:center;justify-content:space-between;padding:0.45rem 0.75rem;background:#EEEDE9;border-bottom:1px solid #DCE0E2;">
    <div style="display:inline-flex;align-items:center;gap:0.45rem;">
      <svg width="18" height="18" viewBox="0 0 24 24" fill="none"><path d="M12 2C8 2 8 4.5 8 4.5V7h4v1H6.5S3 8 3 12.5 5.5 17 5.5 17H8v-3.5S7.8 11 10 11h4s2 .1 2-2V4.5S16 2 12 2zm-1 1.8a.9.9 0 110 1.8.9.9 0 010-1.8z" fill="#5A6F77"/><path d="M12 22c4 0 4-2.5 4-2.5V17h-4v-1h5.5s3.5 0 3.5-4.5S18.5 7 18.5 7H16v3.5s.2 2.5-2 2.5h-4s-2-.1-2 2v4.5S8 22 12 22zm1-1.8a.9.9 0 110-1.8.9.9 0 010 1.8z" fill="none" stroke="#5A6F77" stroke-width="1.5"/></svg>
      <span style="font-size:0.72rem;font-weight:600;text-transform:uppercase;letter-spacing:0.1em;color:#5A6F77;">Python</span>
    </div>
    <button style="display:inline-flex;align-items:center;gap:0.3rem;background:none;border:none;color:#5A6F77;cursor:pointer;padding:0.2rem 0.4rem;border-radius:0.25rem;font-size:0.7rem;font-weight:500;" onclick="(function(b){var c=Array.prototype.map.call(b.closest('.code-card').querySelectorAll('.line'),function(l){return l.textContent.replace(/\u00a0/g,' ').replace(/\s+$/,'')}).join('\n'),t=document.createElement('textarea');t.value=c;t.style.position='fixed';t.style.opacity='0';document.body.appendChild(t);t.select();var ok=false;try{ok=document.execCommand('copy')}catch(e){}document.body.removeChild(t);if(ok){var l=b.querySelector('.copy-label'),ci=b.querySelector('.copy-icon'),chi=b.querySelector('.check-icon');ci.style.display='none';chi.style.display='';l.textContent='Copied!';b.style.color='#2272B4';setTimeout(function(){ci.style.display='';chi.style.display='none';l.textContent='Copy';b.style.color='#5A6F77'},2000)}})(this)">
      <svg class="copy-icon" fill="none" stroke="currentColor" viewBox="0 0 24 24" style="width:14px;height:14px" stroke-width="2" stroke-linecap="round" stroke-linejoin="round"><rect x="9" y="9" width="13" height="13" rx="2"/><path d="M5 15H4a2 2 0 01-2-2V4a2 2 0 012-2h9a2 2 0 012 2v1"/></svg>
      <svg class="check-icon" style="display:none;width:14px;height:14px" fill="none" stroke="currentColor" viewBox="0 0 24 24" stroke-width="2.5" stroke-linecap="round" stroke-linejoin="round"><polyline points="20 6 9 17 4 12"/></svg>
      <span class="copy-label">Copy</span>
    </button>
  </div>
    <div class="f2-code" id="f2-code" style="margin:0;padding:1rem;overflow-x:auto;font-family:'SF Mono','Cascadia Code','Fira Code',Consolas,monospace;font-size:0.82rem;line-height:1.55;color:#1B3139;background:transparent;">
      <span class="line" data-g="0"><span class="tk-cmt"># Find error traces</span></span>
      <span class="line" data-g="0">error_traces = mlflow.<span class="tk-fn">search_traces</span>(</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;locations=[experiment_id],</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;filter_string=<span class="tk-str">f"trace.status = 'ERROR' AND trace.timestamp_ms > {one_hour_ago_ms}"</span>,</span>
      <span class="line" data-g="0">&nbsp;&nbsp;&nbsp;&nbsp;max_results=<span class="tk-num">100</span></span>
      <span class="line" data-g="0">)</span>
      <span class="line" data-g="0"><span class="tk-fn">print</span>(<span class="tk-str">f"Error traces found: {</span><span class="tk-fn">len</span>(error_traces)<span class="tk-str">}"</span>)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="1"><span class="tk-cmt"># Find slow traces (over 5 seconds)</span></span>
      <span class="line" data-g="1">slow_traces = mlflow.<span class="tk-fn">search_traces</span>(</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;locations=[experiment_id],</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;filter_string=<span class="tk-str">f"trace.execution_time_ms > 5000 AND trace.timestamp_ms > {one_hour_ago_ms}"</span>,</span>
      <span class="line" data-g="1">&nbsp;&nbsp;&nbsp;&nbsp;max_results=<span class="tk-num">100</span></span>
      <span class="line" data-g="1">)</span>
      <span class="line" data-g="1"><span class="tk-fn">print</span>(<span class="tk-str">f"Slow traces (>5s): {</span><span class="tk-fn">len</span>(slow_traces)<span class="tk-str">}"</span>)</span>
      <span class="line">&nbsp;</span>
      <span class="line" data-g="2"><span class="tk-cmt"># Find traces by a span attribute set in C1</span></span>
      <span class="line" data-g="2">version_traces = mlflow.<span class="tk-fn">search_traces</span>(</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;locations=[experiment_id],</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;filter_string=<span class="tk-str">f"span.attributes.app_version = '1.2.0' AND trace.timestamp_ms > {one_hour_ago_ms}"</span>,</span>
      <span class="line" data-g="2">&nbsp;&nbsp;&nbsp;&nbsp;max_results=<span class="tk-num">100</span></span>
      <span class="line" data-g="2">)</span>
      <span class="line" data-g="2"><span class="tk-fn">print</span>(<span class="tk-str">f"Traces from app version 1.2.0: {</span><span class="tk-fn">len</span>(version_traces)<span class="tk-str">}"</span>)</span>
    </div>
  </div>
  <div class="f2-explain">
    <div class="f2-explain-card" id="f2-explain-card"></div>
  </div>
</div>

</div>

<script>
var F2_DATA = [
  { color:'#FE3722', title:'Filter by Error Status',
    text:'Uses <code>filter_string="trace.status = \'ERROR\'"</code> to return only failed traces. The <code>trace.*</code> prefix is required for trace-level fields like <code>trace.status</code>, <code>trace.timestamp_ms</code>, and <code>trace.execution_time_ms</code>. You can also filter on <code>metadata.*</code>, <code>tag.*</code>, and <code>span.attributes.*</code> using <code>=</code>, <code>!=</code>, <code>&lt;</code>, <code>&gt;</code>, <code>LIKE</code>, and <code>AND</code>.' },
  { color:'#F8A805', title:'Filter by Execution Time',
    text:'Uses <code>filter_string="trace.execution_time_ms > 5000"</code> to find traces taking over 5 seconds. You can combine filters with <code>AND</code>, e.g. <code>trace.status = \'OK\' AND trace.execution_time_ms > 2500</code>.' },
  { color:'#2272B4', title:'Filter by Span Attribute',
    text:'Uses <code>span.attributes.app_version</code> to find traces with a span that has <code>app_version</code> set to <code>1.2.0</code>, the attribute added in C1. Span attribute filters work for traces stored in Unity Catalog.' }
];
var f2Current = null, f2Light = false;
function f2Toggle() {
  f2Light = !f2Light;
}
function f2Select(id) {
  var code = document.getElementById('f2-code'), card = document.getElementById('f2-explain-card');
  document.querySelectorAll('.f2-card').forEach(function(b){ b.classList.toggle('active', parseInt(b.dataset.id)===id); });
  if (f2Current === id) { code.classList.remove('has-highlight'); card.classList.remove('visible'); document.querySelectorAll('.f2-card').forEach(function(b){b.classList.remove('active');}); f2Current=null; return; }
  f2Current = id; var c = F2_DATA[id];
  code.classList.add('has-highlight');
  code.querySelectorAll('.line').forEach(function(ln){ ln.classList.toggle('hl', ln.dataset.g===String(id)); });
  card.innerHTML = '<div style="height:4px;border-radius:2px;margin:-2px 0 12px;background:' + c.color + ';"></div><div style="font-size:14pt;font-weight:700;margin-bottom:10px;color:#0b2026;">'+c.title+'</div><div>'+c.text+'</div>';
  card.classList.add('visible');
}
f2Select(0);
</script>

In [0]:
# Find error traces
error_traces = mlflow.search_traces(
    locations=[experiment_id],
    filter_string=f"trace.status = 'ERROR' AND trace.timestamp_ms > {one_hour_ago_ms}",
    max_results=100
)
print(f"Error traces found: {len(error_traces)}")

# Find slow traces (over 5 seconds)
slow_traces = mlflow.search_traces(
    locations=[experiment_id],
    filter_string=f"trace.execution_time_ms > 5000 AND trace.timestamp_ms > {one_hour_ago_ms}",
    max_results=100
)
print(f"Slow traces (>5s): {len(slow_traces)}")

# Find traces by a span attribute set in C1
version_traces = mlflow.search_traces(
    locations=[experiment_id],
    filter_string=f"span.attributes.app_version = '1.2.0' AND trace.timestamp_ms > {one_hour_ago_ms}",
    max_results=100
)
print(f"Traces from app version 1.2.0: {len(version_traces)}")

### E3. SQL-Based Trace Queries

Because the traces are stored in Unity Catalog, you can also query them with SQL. MLflow creates two views over the trace tables: `prod_tracing_trace_unified`, with full span data, and `prod_tracing_trace_metadata`, with tags and assessments only, which is faster when you don't need span data. The query below needs each trace's duration, so it uses the unified view to count the turns in each session and their average duration.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Use <code>mlflow.search_traces()</code> for interactive analysis in Python. Use SQL on the trace views for dashboards and scheduled reports.</p>
</div>
</div>
</div>

In [0]:
display(spark.sql(f"""
    SELECT trace_metadata:['mlflow.trace.session']::STRING AS session_id,
           COUNT(*) AS turns,
           ROUND(AVG(execution_duration_ms)) AS avg_duration_ms
    FROM {catalog_name}.{schema_name}.prod_tracing_trace_unified
    WHERE trace_metadata:['mlflow.trace.session'] IS NOT NULL
    GROUP BY 1
"""))

## F. Conclusion

1. **Unity Catalog trace storage** (`trace_location=UnityCatalog(...)`) keeps traces in governed Delta tables you can query with Python or SQL
2. **Manual tracing** (`@mlflow.trace`) with semantic `span_type` labels adds visibility into custom logic: tool calls, retrieval, business logic
3. **Span attributes** attach production context (app version, turn number) that is critical for debugging; **trace metadata** (`session_id`, `user`) enables session grouping
4. **Multi-turn tracing** uses `mlflow.update_current_trace()` with a shared `session_id` to group conversation turns in MLflow's Sessions view
5. **Trace search** (`mlflow.search_traces()`) filters by status, timing, and span attributes, with a time range to limit each scan
6. **SQL queries** on the trace views support dashboards and scheduled reports

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
